In [1200]:
board = [
[0,0,0],
[0,0,0],
[0,0,0]
]



In [1201]:
board

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]

In [1202]:
board[1][1] = 1

In [1203]:
board

[[0, 0, 0], [0, 1, 0], [0, 0, 0]]

In [1204]:
empty = 0
X = 1 
O = -1

In [1205]:
board[0][0] =X

In [1206]:
board[1][1] = O

In [1207]:
board

[[1, 0, 0], [0, -1, 0], [0, 0, 0]]

In [1208]:
def invalid(a,b):
    if board[a][b] == X or board[a][b] == O:
        return True

    return False

In [1209]:
def move(current_player,a,b):
  if current_player == X:
    board[a][b] = 1
  else:
     board[a][b] = -1


In [1210]:
def clear(board):
  for i in range (3):
    for j in range (3):
      board[i][j] = 0

In [1211]:
clear(board)

In [1212]:
board  

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]

In [1213]:
current_player = X
move(X,0,0)

In [1214]:
board

[[1, 0, 0], [0, 0, 0], [0, 0, 0]]

In [1215]:
board

[[1, 0, 0], [0, 0, 0], [0, 0, 0]]

In [1216]:
def check_win(board):
  if board[0][0] == board[0][1] == board [0][2] == 1 or board[1][0] == board[1][1] == board [1][2] == 1 or board[2][0] == board[2][1] == board [2][2] == 1:
    return X
  elif board[0][0] == board[1][0] == board [2][0] == 1 or board[0][1] == board[1][1] == board [2][1] == 1 or board[0][2] == board[1][2] == board [2][2] == 1:
    return X
  elif board[0][0] == board[1][1] == board[2][2] == 1:
    return X
  elif board[0][2] == board[1][1] == board[2][0] == 1:
    return X
  elif board[0][0] == board[0][1] == board [0][2] == -1 or board[1][0] == board[1][1] == board [1][2] == -1 or board[2][0] == board[2][1] == board [2][2] == -1:
    return O
  elif board[0][0] == board[1][0] == board [2][0] == -1 or board[0][1] == board[1][1] == board [2][1] == -1 or board[0][2] == board[1][2] == board [2][2] == -1:
    return O
  elif board[0][0] == board[1][1] == board[2][2] == -1:
    return O
  elif board[0][2] == board[1][1] == board[2][0] == -1:
    return O
  else:
    return None

In [1217]:
Draw = "Draw"
def check_draw(board):
  board_full = True
  for i in range (3):
    for j in range (3):
      if board[i][j] == 0:
        board_full= False
 
  if check_win(board) == None and board_full:
    return Draw
  

In [1218]:
import random
def random_move():

  row = random.randint(0,2)
  column = random.randint(0,2)
  return row,column

In [1219]:
clear(board)
print(board)


[[0, 0, 0], [0, 0, 0], [0, 0, 0]]


In [1220]:
def flatten(board):
  new_board = []
  for i in range (3):
    for j in range (3):
      new_board.append(board[i][j])
  return new_board 


In [1221]:
def indexing(row,column):
  index = row*3 + column
  return index


In [1222]:
X_train = []
y_train = []


In [1223]:
for game in range (5000):
  clear(board)
  history = []
  current_player = X
  while (check_win(board)!= X and check_win(board)!= O) and check_draw(board)!=Draw:
    board_snapshot = [row[:] for row in board]
    row , column = random_move()

    while invalid(row,column):
      row , column = random_move()
    if current_player == X:
      history.append([board_snapshot,
                (row,column),
                X])
      move(X, row, column)
      
      current_player = O

    else:
      history.append([board_snapshot,
                (row,column),
                O])
      move(O,row,column)
      
      current_player = X

  if check_win(board) == X:
    reward = 1

  elif check_win(board) == O:
    reward = -1

  elif check_draw(board) == Draw:
    reward = 0
  for i in history:
    i.append(reward)
  winner_moves = []
  loser_moves = []
  for i in range (len(history)):
    if (reward==1 and history[i][2] == 1):
      winner_moves.append(history[i])
    elif (reward==-1 and history[i][2] == -1):
       winner_moves.append(history[i])
      
    if (reward == 1) and history[i][2] == -1:
      loser_moves.append(history[i])
    elif reward == -1 and history[i][2] == 1:
      loser_moves.append(history[i])

  if len(winner_moves) > 0:
    last_move = winner_moves[-3:]

    for moves in last_move:
      X_train.append(flatten(moves[0]))
      y_train.append(indexing(*moves[1]))  

  if len(loser_moves) > 0:

    loser_move = loser_moves[-1]

    X_train.append(flatten(loser_move[0]))
    y_train.append(indexing(*loser_move[1]))
    
  



In [1224]:
#y_train

In [1225]:
print(len(X_train))
print(len(y_train))

17680
17680


In [1226]:
import torch

In [1227]:
X_train = torch.tensor(X_train)
y_train = torch.tensor(y_train)

In [1228]:
print(X_train.shape)
print(y_train.shape)

torch.Size([17680, 9])
torch.Size([17680])


# Defining the model

In [1229]:
# define NN class
import torch.nn as nn
class MyNN(nn.Module):
  def __init__(self, num_features):

    super().__init__()
    self.model = nn.Sequential(
      nn.Linear(num_features, 32),
      nn.ReLU(),
      nn.Linear(32,32),
      nn.ReLU(),
      nn.Linear(32, 9)
    )

  def forward(self, x):
      return self.model(x)

  

In [1230]:
model = MyNN(9)

In [1231]:
criterion = nn.CrossEntropyLoss()

In [1232]:
import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [1233]:
X_train = X_train.float()
epochs = 100

# training loop

In [1234]:
for epoch in range(epochs):
  y_pred = model(X_train)
  loss = criterion(y_pred, y_train)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')


Epoch: 1, Loss: 2.209125518798828
Epoch: 2, Loss: 2.2077906131744385
Epoch: 3, Loss: 2.2065131664276123
Epoch: 4, Loss: 2.2052927017211914
Epoch: 5, Loss: 2.20412540435791
Epoch: 6, Loss: 2.2030086517333984
Epoch: 7, Loss: 2.201935052871704
Epoch: 8, Loss: 2.2009053230285645
Epoch: 9, Loss: 2.1999151706695557
Epoch: 10, Loss: 2.198965311050415
Epoch: 11, Loss: 2.1980478763580322
Epoch: 12, Loss: 2.197155475616455
Epoch: 13, Loss: 2.196288585662842
Epoch: 14, Loss: 2.195446491241455
Epoch: 15, Loss: 2.1946229934692383
Epoch: 16, Loss: 2.1938185691833496
Epoch: 17, Loss: 2.193028211593628
Epoch: 18, Loss: 2.1922523975372314
Epoch: 19, Loss: 2.191493272781372
Epoch: 20, Loss: 2.1907477378845215
Epoch: 21, Loss: 2.1900126934051514
Epoch: 22, Loss: 2.1892898082733154
Epoch: 23, Loss: 2.188575029373169
Epoch: 24, Loss: 2.1878674030303955
Epoch: 25, Loss: 2.1871681213378906
Epoch: 26, Loss: 2.186473846435547
Epoch: 27, Loss: 2.1857850551605225
Epoch: 28, Loss: 2.185103178024292
Epoch: 29, Los

In [1235]:
sample = X_train[0]

In [1236]:
sample = sample.unsqueeze(0)

In [1237]:
output = model(sample)
output

tensor([[-0.1070,  0.0346,  0.2309,  0.0602,  0.2733,  0.0480, -0.0661, -0.0189,
          0.1201]], grad_fn=<AddmmBackward0>)

In [1238]:
prediction = torch.argmax(output)
print(prediction)

tensor(4)


In [1239]:
clear(board)
print(board)

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]


In [1242]:

ai_wins = 0
random_wins = 0
draws = 0

for game in range(100):
  clear(board)
  current_player = X
  while (check_win(board)!=X and check_win(board)!=O) and check_draw(board)!=Draw:
    if current_player == X:
      input = torch.tensor(flatten(board))
      input = input.float()
      input = input.unsqueeze(0)
      output = model(input)

      ranked_moves = torch.argsort(output, descending=True)

      for index in ranked_moves[0]:
        index = index.item()
        row = index // 3
        column = index % 3
        if not invalid(row,column):
         move(X,row,column)
        current_player = O
        break
    
    else:

      row,column = random_move()

      while invalid(row,column):
        row,column = random_move()
      
      move(O,row,column)

      current_player = X
    
    
  if check_win(board) == X:
    ai_wins += 1

  elif check_win(board) == O:
    random_wins += 1

  else:
    draws += 1

In [1243]:

print(ai_wins)
print(random_wins)
print(draws)

33
67
0
